# Qwen Sentiment Base vs QLoRA Evaluation

这个 Notebook 会从真实数据中抽取 100 条样本，同时对比：

- `base-only`：只加载原始 Qwen 模型
- `base + adapter`：加载 QLoRA adapter

最终输出：

- 样本标签分布
- 每条样本的双模型预测结果
- 两个模型各自的混淆矩阵
- accuracy / precision / recall / f1 对比
- 每个类别的 precision / recall / f1 / support 对比

In [28]:
import os
from pathlib import Path

import pandas as pd
import torch
from peft import PeftModel
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

cwd = Path.cwd().resolve()
if cwd.name in {"nasdaq_news_sentiment", "risk_nasdaq", "Finance"}:
    REPO_ROOT = cwd.parent
else:
    REPO_ROOT = cwd

BASE_MODEL_PATH = Path(os.getenv("QWEN_MODEL_PATH", REPO_ROOT / "models" / "Qwen3-0.6B"))
ADAPTER_PATH = Path(os.getenv("SENTIMENT_ADAPTER_PATH", REPO_ROOT / "models" / "qwen_sentiment_model_qlora"))
DATA_PATH = Path(os.getenv("SENTIMENT_TEST_DATA_PATH", REPO_ROOT / "data" / "nasdaq_news_sentiment" / "sentiment_deepseek_new_cleaned_nasdaq_news_full.csv"))
SAMPLE_SIZE = int(os.getenv("SENTIMENT_EVAL_SAMPLE_SIZE", "100"))
START_OFFSET = int(os.getenv("SENTIMENT_EVAL_START_OFFSET", "10000"))
LABELS = [1, 2, 3, 4, 5]

print("BASE_MODEL_PATH =", BASE_MODEL_PATH)
print("ADAPTER_PATH    =", ADAPTER_PATH)
print("DATA_PATH       =", DATA_PATH)
print("SAMPLE_SIZE     =", SAMPLE_SIZE)
print("START_OFFSET    =", START_OFFSET)
print("CUDA available  =", torch.cuda.is_available())

if not BASE_MODEL_PATH.exists():
    raise FileNotFoundError(f"Base model path not found: {BASE_MODEL_PATH}")
if not ADAPTER_PATH.exists():
    raise FileNotFoundError(f"Adapter path not found: {ADAPTER_PATH}")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset path not found: {DATA_PATH}")

BASE_MODEL_PATH = /home/irving/workspace/agent_demo/models/Qwen3-0.6B
ADAPTER_PATH    = /home/irving/workspace/agent_demo/models/qwen_sentiment_model_qlora
DATA_PATH       = /home/irving/workspace/agent_demo/data/nasdaq_news_sentiment/sentiment_deepseek_new_cleaned_nasdaq_news_full.csv
SAMPLE_SIZE     = 100
START_OFFSET    = 10000
CUDA available  = True


In [29]:
def get_quant_config():
    compute_dtype = torch.float16
    if torch.cuda.is_available():
        major, _ = torch.cuda.get_device_capability()
        if major >= 8:
            compute_dtype = torch.bfloat16

    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )


def create_sentiment_test_prompt(text, stock_symbol="STOCK"):
    system_prompt = (
        "Forget all your previous instructions. You are a financial expert with stock "
        "recommendation experience. Based on a specific stock, score for range from 1 "
        "to 5, where 1 is negative, 2 is somewhat negative, 3 is neutral, 4 is somewhat "
        "positive, 5 is positive. 1 summarized news will be passed in each time, you "
        "will give score in format as shown below in the response from assistant."
    )

    user_content = f"News to Stock Symbol -- {stock_symbol}: {text}"

    return f"""System: {system_prompt}

User: News to Stock Symbol -- AAPL: Apple (AAPL) increase 22%
Assistant: 5

User: News to Stock Symbol -- AAPL: Apple (AAPL) price decreased 30%
Assistant: 1

User: News to Stock Symbol -- AAPL: Apple (AAPL) announced iPhone 15
Assistant: 4

User: {user_content}
Assistant:"""


def extract_score(generated_text):
    assistant_response = generated_text.split("Assistant:")[-1].strip()
    try:
        score = int(assistant_response.split()[0])
        if score in LABELS:
            return score
    except Exception:
        pass
    return None


def run_prediction(model, tokenizer, text, stock_symbol="STOCK", max_length=512):
    prompt = create_sentiment_test_prompt(text, stock_symbol)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predicted_score = extract_score(generated_text)
    return generated_text, predicted_score


def build_confusion_df(y_true, y_pred, labels=LABELS):
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    return pd.DataFrame(matrix, index=[f"true_{x}" for x in labels], columns=[f"pred_{x}" for x in labels])


def build_metrics_summary(y_true, y_pred):
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, average="weighted", zero_division=0)
    micro_p, micro_r, micro_f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, average="micro", zero_division=0)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": macro_p,
        "recall_macro": macro_r,
        "f1_macro": macro_f1,
        "precision_weighted": weighted_p,
        "recall_weighted": weighted_r,
        "f1_weighted": weighted_f1,
        "precision_micro": micro_p,
        "recall_micro": micro_r,
        "f1_micro": micro_f1,
    }


def build_per_class_metrics_df(y_true, y_pred, model_name):
    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, average=None, zero_division=0)
    return pd.DataFrame({
        "label": LABELS,
        f"{model_name}_precision": precision,
        f"{model_name}_recall": recall,
        f"{model_name}_f1": f1,
        f"{model_name}_support": support,
    })


In [30]:
quant_config = get_quant_config()

tokenizer = AutoTokenizer.from_pretrained(str(BASE_MODEL_PATH), trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Loading base-only model...")
base_only_model = AutoModelForCausalLM.from_pretrained(
    str(BASE_MODEL_PATH),
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
)
base_only_model.eval()

print("Loading base + adapter model...")
adapter_base_model = AutoModelForCausalLM.from_pretrained(
    str(BASE_MODEL_PATH),
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
)
adapter_model = PeftModel.from_pretrained(adapter_base_model, str(ADAPTER_PATH))
adapter_model.eval()

print("Models loaded.")
print("If this cell OOMs on your GPU, you need to evaluate the two paths sequentially instead of keeping both in memory.")

Loading base-only model...
Loading base + adapter model...
Models loaded.
If this cell OOMs on your GPU, you need to evaluate the two paths sequentially instead of keeping both in memory.


In [31]:
raw_df = pd.read_csv(DATA_PATH)
eval_df = raw_df[raw_df["Lsa_summary"].notna() & raw_df["sentiment_deepseek"].notna()].copy()
eval_df = eval_df[eval_df["sentiment_deepseek"] != 0].iloc[START_OFFSET:START_OFFSET + SAMPLE_SIZE].copy()
eval_df["sentiment_deepseek"] = eval_df["sentiment_deepseek"].astype(int)

print(f"Selected rows: {len(eval_df)}")
print(f"Using effective rows from offset {START_OFFSET} to {START_OFFSET + len(eval_df) - 1}")
eval_df[["Stock_symbol", "sentiment_deepseek", "Lsa_summary"]].head(3)

Selected rows: 100
Using effective rows from offset 10000 to 10099


,Stock_symbol,sentiment_deepseek,Lsa_summary
11082,AMD,2,Components of that ETF with the highest volume...
11083,AMD,2,"Moreover, if this Intel/AMD chip poses a serio..."
11084,AMD,2,Aside from the fact that NVDA stock is priced ...


In [32]:
label_distribution_df = (
    eval_df["sentiment_deepseek"]
    .value_counts()
    .reindex(LABELS, fill_value=0)
    .rename_axis("label")
    .reset_index(name="count")
)
label_distribution_df

,label,count
0,1,3
1,2,25
2,3,39
3,4,33
4,5,0


In [33]:
rows = []
for idx, row in eval_df.reset_index(drop=True).iterrows():
    text = row["Lsa_summary"]
    stock_symbol = row.get("Stock_symbol", "STOCK")
    true_label = int(row["sentiment_deepseek"])

    base_raw, base_pred = run_prediction(base_only_model, tokenizer, text, stock_symbol)
    adapter_raw, adapter_pred = run_prediction(adapter_model, tokenizer, text, stock_symbol)

    rows.append({
        "row_id": idx,
        "stock_symbol": stock_symbol,
        "true_sentiment": true_label,
        "base_pred": base_pred,
        "adapter_pred": adapter_pred,
        "base_correct": base_pred == true_label if base_pred is not None else False,
        "adapter_correct": adapter_pred == true_label if adapter_pred is not None else False,
        "text": text,
        "base_raw": base_raw,
        "adapter_raw": adapter_raw,
    })

compare_df = pd.DataFrame(rows)
print(f"Finished predictions for {len(compare_df)} rows.")
compare_df[["row_id", "stock_symbol", "true_sentiment", "base_pred", "adapter_pred", "base_correct", "adapter_correct"]].head(20)

/root/miniconda3/envs/lamost/lib/python3.13/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/root/miniconda3/envs/lamost/lib/python3.13/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/root/miniconda3/envs/lamost/lib/python3.13/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Finished predictions for 100 rows.


,row_id,stock_symbol,true_sentiment,base_pred,adapter_pred,base_correct,adapter_correct
0,0,AMD,2,3.0,3,False,False
1,1,AMD,2,3.0,3,False,False
2,2,AMD,2,3.0,2,False,True
3,3,AMD,3,3.0,3,True,True
4,4,AMD,3,3.0,3,True,True
5,5,AMD,2,3.0,3,False,False
6,6,AMD,4,3.0,3,False,False
7,7,AMD,2,3.0,2,False,True
8,8,AMD,1,3.0,3,False,False
9,9,AMD,3,3.0,3,True,True


In [34]:
results_df = compare_df.copy()
results_df["base_pred_filled"] = results_df["base_pred"].fillna(0).astype(int)
results_df["adapter_pred_filled"] = results_df["adapter_pred"].fillna(0).astype(int)

valid_base_df = results_df[results_df["base_pred"].isin(LABELS)].copy()
valid_adapter_df = results_df[results_df["adapter_pred"].isin(LABELS)].copy()

print("base parsed rows   =", len(valid_base_df))
print("adapter parsed rows=", len(valid_adapter_df))

base parsed rows   = 93
adapter parsed rows= 100


In [35]:
overall_metrics_df = pd.DataFrame([
    {"model": "base_only", **build_metrics_summary(valid_base_df["true_sentiment"], valid_base_df["base_pred"])},
    {"model": "base_plus_adapter", **build_metrics_summary(valid_adapter_df["true_sentiment"], valid_adapter_df["adapter_pred"])},
]).round(4)
overall_metrics_df

,model,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,precision_micro,recall_micro,f1_micro
0,base_only,0.3656,0.0782,0.1943,0.1115,0.1471,0.3656,0.2098,0.3656,0.3656,0.3656
1,base_plus_adapter,0.5600,0.3427,0.3201,0.3043,0.5520,0.5600,0.5195,0.5600,0.5600,0.5600


In [36]:
base_confusion_df = build_confusion_df(valid_base_df["true_sentiment"], valid_base_df["base_pred"])
base_confusion_df

,pred_1,pred_2,pred_3,pred_4,pred_5
true_1,0,0,3,0,0
true_2,0,0,21,0,2
true_3,0,0,34,0,1
true_4,0,0,29,0,3
true_5,0,0,0,0,0


In [37]:
adapter_confusion_df = build_confusion_df(valid_adapter_df["true_sentiment"], valid_adapter_df["adapter_pred"])
adapter_confusion_df

,pred_1,pred_2,pred_3,pred_4,pred_5
true_1,0,1,2,0,0
true_2,0,4,15,6,0
true_3,0,1,29,9,0
true_4,0,1,8,23,1
true_5,0,0,0,0,0


In [38]:
per_class_metrics_df = build_per_class_metrics_df(valid_base_df["true_sentiment"], valid_base_df["base_pred"], "base")
per_class_metrics_df = per_class_metrics_df.merge(
    build_per_class_metrics_df(valid_adapter_df["true_sentiment"], valid_adapter_df["adapter_pred"], "adapter"),
    on="label",
    how="outer",
)
per_class_metrics_df.round(4)

,label,base_precision,base_recall,base_f1,base_support,adapter_precision,adapter_recall,adapter_f1,adapter_support
0,1,0.0000,0.0000,0.0000,3,0.0000,0.0000,0.0000,3
1,2,0.0000,0.0000,0.0000,23,0.5714,0.1600,0.2500,25
2,3,0.3908,0.9714,0.5574,35,0.5370,0.7436,0.6237,39
3,4,0.0000,0.0000,0.0000,32,0.6053,0.6970,0.6479,33
4,5,0.0000,0.0000,0.0000,0,0.0000,0.0000,0.0000,0


In [39]:
prediction_distribution_df = pd.DataFrame({
    "label": LABELS,
    "true_count": results_df["true_sentiment"].value_counts().reindex(LABELS, fill_value=0).values,
    "base_pred_count": valid_base_df["base_pred"].value_counts().reindex(LABELS, fill_value=0).values,
    "adapter_pred_count": valid_adapter_df["adapter_pred"].value_counts().reindex(LABELS, fill_value=0).values,
})
prediction_distribution_df

,label,true_count,base_pred_count,adapter_pred_count
0,1,3,0,0
1,2,25,0,7
2,3,39,87,54
3,4,33,0,38
4,5,0,6,1


In [40]:
compare_df[[
    "row_id",
    "stock_symbol",
    "true_sentiment",
    "base_pred",
    "adapter_pred",
    "base_correct",
    "adapter_correct",
    "text",
    "base_raw",
    "adapter_raw",
]].head(30)

,row_id,stock_symbol,true_sentiment,base_pred,adapter_pred,base_correct,adapter_correct,text,base_raw,adapter_raw
0,0,AMD,2,3.0,3,False,False,Components of that ETF with the highest volume...,System: Forget all your previous instructions....,System: Forget all your previous instructions....
1,1,AMD,2,3.0,3,False,False,"Moreover, if this Intel/AMD chip poses a serio...",System: Forget all your previous instructions....,System: Forget all your previous instructions....
2,2,AMD,2,3.0,2,False,True,Aside from the fact that NVDA stock is priced ...,System: Forget all your previous instructions....,System: Forget all your previous instructions....
3,3,AMD,3,3.0,3,True,True,Chip rivals Advanced Micro Devices ( AMD ) and...,System: Forget all your previous instructions....,System: Forget all your previous instructions....
4,4,AMD,3,3.0,3,True,True,The PowerShares FTSE RAFI US 1500 Small-Mid Po...,System: Forget all your previous instructions....,System: Forget all your previous instructions....
5,5,AMD,2,3.0,3,False,False,My conviction in this is strengthened by looki...,System: Forget all your previous instructions....,System: Forget all your previous instructions....
6,6,AMD,4,3.0,3,False,False,We've seen numerous such sell-offs in AMD stoc...,System: Forget all your previous instructions....,System: Forget all your previous instructions....
7,7,AMD,2,3.0,2,False,True,"Moreover, a decline in the shares of Advanced ...",System: Forget all your previous instructions....,System: Forget all your previous instructions....
8,8,AMD,1,3.0,3,False,False,Analysts continue to assess Advanced Micro Dev...,System: Forget all your previous instructions....,System: Forget all your previous instructions....
9,9,AMD,3,3.0,3,True,True,More from InvestorPlace 5 Battered Stocks That...,System: Forget all your previous instructions....,System: Forget all your previous instructions....


如果你想改样本数，直接调整环境变量 `SENTIMENT_EVAL_SAMPLE_SIZE` 或修改 `SAMPLE_SIZE`。

如果你想改起始位置，调整环境变量 `SENTIMENT_EVAL_START_OFFSET`。当前默认从第 10001 条有效样本开始取。 

如果双模型同时加载显存不够，就把 Notebook 改成先跑 `base-only` 保存结果，再释放显存后跑 `adapter`。当前版本是为了满足直接对比而保留双模型同时在显存中。